# 4.5 Lab - Deploy a DAB to Multiple Environments
Starter: **`04-automated-deployment-dabs/labs/4.5_multi_env_dab/`** (a copy of the 4.4 bundle). Solution: `solutions/4.5_multi_env_dab/`.

| TODO | What |
|---|---|
| 1 | Variable `alert_email` **without** a default |
| 2 | Set `alert_email` in `dev` and `stage` |
| 3 | New target `prod`: production mode, shared root path, schema `orders`, 1000 rows per run, `alert_email: oncall@example.com`, group `users` gets `CAN_VIEW` |
| 4 | Job emails `${var.alert_email}` on failure |

In [ ]:
from adelab import bundles
from adelab.spark import PROJECT_ROOT

LAB = PROJECT_ROOT / "04-automated-deployment-dabs" / "labs" / "4.5_multi_env_dab"
status = bundles.workspace_status()

## CHECK (offline)

In [ ]:
assert not bundles.validate_schema(LAB), "fix the schema errors above"
cfg = bundles.load_bundle(LAB)
assert "alert_email" in cfg["variables"] and "default" not in (cfg["variables"]["alert_email"] or {}), "TODO 1"
expected = {"dev": None, "stage": "data-team@example.com", "prod": "oncall@example.com"}
for target, email in expected.items():
    r = bundles.preview(LAB, target, show=False)          # fails if alert_email has no value for the target
    job = r["resources"]["jobs"]["orders_job"]
    got = job.get("email_notifications", {}).get("on_failure", [])
    assert got and (email is None or got == [email]), f"TODO 2/4 for {target}: {got}"
prod = bundles.preview(LAB, "prod", show=False)
assert prod["bundle"]["mode"] == "production", "TODO 3: production mode"
assert prod["resources"]["pipelines"]["orders_pipeline"]["schema"] == "orders", "TODO 3: schema 'orders' in prod"
assert prod["resources"]["jobs"]["orders_job"]["tasks"][0]["notebook_task"]["base_parameters"]["rows"] == "1000",     "TODO 3: orders_per_run 1000 in prod"
assert prod.get("workspace", {}).get("root_path", "").startswith("/Workspace/Shared/"), "TODO 3: shared root_path"
assert {"group_name": "users", "level": "CAN_VIEW"} in prod.get("permissions", []), "TODO 3: permissions"
print("All offline checks passed")

## Deploy (workspace)

In [ ]:
if bundles.require_workspace(status):
    for t in ("dev", "prod"):
        bundles.cli("bundle", "validate", "-t", t, cwd=LAB)
    bundles.cli("bundle", "deploy", "-t", "prod", cwd=LAB)
    bundles.cli("bundle", "summary", "-t", "prod", cwd=LAB)

In [ ]:
if bundles.require_workspace(status):
    bundles.cli("bundle", "destroy", "-t", "prod", "--auto-approve", cwd=LAB)